In [2]:
import os
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from sec_edgar_downloader import Downloader
import time

In [3]:
# Define download directories
DOWNLOAD_DIR = "sec-edgar-filings"
FOLDER_1 = "C:/Users/sudet/Desktop/sec-edgar-filings"
FOLDER_2 = "C:/Users/sudet/Desktop/to-be-processed"

In [4]:
# Load tickers from CSV
file_path = "companies.csv"  # Update this path if needed
companies_df = pd.read_csv(file_path, header=None)
companies_df = companies_df.iloc[1500:2000]
tickers = companies_df.iloc[1:, 1].tolist()  # Extract tickers (ignoring header)
cik_dict = {row[1]: str(row[0]).zfill(10) for _, row in companies_df.iloc[1:].iterrows()}  # Map Ticker -> Padded CIK

In [5]:
import sqlite3

def filing_exists(ticker, form, cik, year):
    """
    Check if the filing for a specific ticker, form, CIK, and year already exists 
    in either FOLDER_1, FOLDER_2, or the database.
    """
    paths_to_check = [
        os.path.join(FOLDER_2, ticker, form)
    ]
    
    cik_str = str(cik).zfill(10)  # Ensures leading zeros (e.g., 320193 -> 0000320193)
    year_suffix = str(year)[-2:]  # Convert year to 2-digit format (e.g., 2017 -> "17")

    print(f"\nChecking for {ticker} ({form}, {year}) - CIK: {cik_str}")  # Debug Print

    for base_path in paths_to_check:
        print(f"📂 Looking in: {base_path}")  # Debug print

        if not os.path.exists(base_path):
            print(f"Path does not exist: {base_path}")
            continue  # Skip to the next folder if this one doesn't exist

        found_folders = os.listdir(base_path)
        print(f"📌 Found {len(found_folders)} folders in {base_path}: {found_folders}")  # Debug print

        for folder in found_folders:
            #print(f"   📁 Checking folder: {folder}")  # Debug print
            
            # Expected format: "0000320193-17-000009"
            if folder.startswith(f"{cik_str}-{year_suffix}"):
                print(f"✅ Match found: {folder} for {ticker} in {base_path}")
                return True  # Found a matching filing

    print(f"No match found for {ticker} in {paths_to_check}")  # Debugging output
    return False  # No matching filing found

In [6]:
years = range(2000, 2025)  # Adjust as needed
forms = ["10-Q"]  # Forms to download

In [7]:
def download_filings(cik, ticker):
    email_address = "hitiras@hotmail.com"  # Replace with your email address
    dl = Downloader(DOWNLOAD_DIR, email_address)

    for form in forms:
        for year in years:
            if filing_exists(ticker, form, cik, year):
                print(f"Skipping {form} for {ticker} ({year}) - Already Downloaded")
                continue

            try:
                print(f"Downloading {form} for {ticker} ({year})...")
                time.sleep(1)  # Avoid SEC rate limits
                dl.get(form, cik, after=f"{year}-01-01", before=f"{year}-12-31")
                #time.sleep(5)  # Avoid SEC rate limits
                
            except Exception as e:
                print(f"Error downloading {form} for {ticker} in {year}: {e}")
                time.sleep(30)  # Sleep longer if an error occurs

In [8]:
def main():
    max_threads = 10  # Reduce concurrency to prevent SEC rate limits
    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        futures = {executor.submit(download_filings, cik_dict[ticker], ticker): ticker for ticker in tickers}

        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading SEC Filings"):
            ticker = futures[future]
            try:
                future.result()  # Ensure exceptions are caught
            except Exception as e:
                print(f"Error downloading {ticker}: {e}")
                time.sleep(30)

    print("Download complete!")


In [9]:
print("Strating SEC filings download...")
if __name__ == "__main__":
    main()

Strating SEC filings download...



Checking for IBOC (10-Q, 2000) - CIK: 0000315709
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\IBOC\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\IBOC\10-Q
No match found for IBOC in ['C:/Users/sudet/Desktop/to-be-processed\\IBOC\\10-Q']

Checking for UNF (10-Q, 2000) - CIK: 0000717954
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\UNF\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\UNF\10-Q
No match found for UNF in ['C:/Users/sudet/Desktop/to-be-processed\\UNF\\10-Q']

Checking for SWTX (10-Q, 2000) - CIK: 0001773427
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\SWTX\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\SWTX\10-Q
No match found for SWTX in ['C:/Users/sudet/Desktop/to-be-processed\\SWTX\\10-Q']

Checking for WFRD (10-Q, 2000) - CIK: 0001603923
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\WFRD\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\WFRD\10-Q
No match found for WFRD in 


Checking for IBOC (10-Q, 2009) - CIK: 0000315709
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\IBOC\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\IBOC\10-Q
No match found for IBOC in ['C:/Users/sudet/Desktop/to-be-processed\\IBOC\\10-Q']

Checking for PAY (10-Q, 2022) - CIK: 0001841156
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\PAY\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\PAY\10-Q
No match found for PAY in ['C:/Users/sudet/Desktop/to-be-processed\\PAY\\10-Q']

Checking for SWTX (10-Q, 2021) - CIK: 0001773427
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\SWTX\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\SWTX\10-Q
No match found for SWTX in ['C:/Users/sudet/Desktop/to-be-processed\\SWTX\\10-Q']

Checking for PRGO (10-Q, 2000) - CIK: 0001585364
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\PRGO\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\PRGO\10-Q
No match found for PRGO in 


Checking for PRGO (10-Q, 2006) - CIK: 0001585364
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\PRGO\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\PRGO\10-Q
No match found for PRGO in ['C:/Users/sudet/Desktop/to-be-processed\\PRGO\\10-Q']

Checking for BCO (10-Q, 2000) - CIK: 0000078890
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\BCO\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\BCO\10-Q
No match found for BCO in ['C:/Users/sudet/Desktop/to-be-processed\\BCO\\10-Q']

Checking for IBOC (10-Q, 2012) - CIK: 0000315709
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\IBOC\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\IBOC\10-Q
No match found for IBOC in ['C:/Users/sudet/Desktop/to-be-processed\\IBOC\\10-Q']

Checking for PRGO (10-Q, 2007) - CIK: 0001585364
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\PRGO\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\PRGO\10-Q
No match found for PRGO in 


Checking for PRGO (10-Q, 2009) - CIK: 0001585364
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\PRGO\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\PRGO\10-Q
No match found for PRGO in ['C:/Users/sudet/Desktop/to-be-processed\\PRGO\\10-Q']

Checking for FROG (10-Q, 2000) - CIK: 0001800667
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\FROG\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\FROG\10-Q
No match found for FROG in ['C:/Users/sudet/Desktop/to-be-processed\\FROG\\10-Q']

Checking for SPR (10-Q, 2014) - CIK: 0001364885
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\SPR\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\SPR\10-Q
No match found for SPR in ['C:/Users/sudet/Desktop/to-be-processed\\SPR\\10-Q']

Checking for BCO (10-Q, 2001) - CIK: 0000078890
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\BCO\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\BCO\10-Q
No match found for BCO in ['C:

...


Checking for PPBI (10-Q, 2021) - CIK: 0001028918
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\PPBI\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\PPBI\10-Q
No match found for PPBI in ['C:/Users/sudet/Desktop/to-be-processed\\PPBI\\10-Q']

Checking for EFSC (10-Q, 2019) - CIK: 0001025835
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\EFSC\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\EFSC\10-Q
No match found for EFSC in ['C:/Users/sudet/Desktop/to-be-processed\\EFSC\\10-Q']

Checking for NBTB (10-Q, 2023) - CIK: 0000790359
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\NBTB\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\NBTB\10-Q
No match found for NBTB in ['C:/Users/sudet/Desktop/to-be-processed\\NBTB\\10-Q']

Checking for AGM (10-Q, 2019) - CIK: 0000845877
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\AGM\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\AGM\10-Q
No match found for AGM in


Checking for AGM (10-Q, 2020) - CIK: 0000845877
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\AGM\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\AGM\10-Q
No match found for AGM in ['C:/Users/sudet/Desktop/to-be-processed\\AGM\\10-Q']

Checking for EFSC (10-Q, 2021) - CIK: 0001025835
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\EFSC\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\EFSC\10-Q
No match found for EFSC in ['C:/Users/sudet/Desktop/to-be-processed\\EFSC\\10-Q']

Checking for PPBI (10-Q, 2023) - CIK: 0001028918
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\PPBI\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\PPBI\10-Q
No match found for PPBI in ['C:/Users/sudet/Desktop/to-be-processed\\PPBI\\10-Q']

Checking for EFSC (10-Q, 2022) - CIK: 0001025835
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\EFSC\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\EFSC\10-Q
No match found for EFSC in 


Checking for EFSC (10-Q, 2023) - CIK: 0001025835
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\EFSC\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\EFSC\10-Q
No match found for EFSC in ['C:/Users/sudet/Desktop/to-be-processed\\EFSC\\10-Q']

Checking for AGM (10-Q, 2022) - CIK: 0000845877
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\AGM\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\AGM\10-Q
No match found for AGM in ['C:/Users/sudet/Desktop/to-be-processed\\AGM\\10-Q']

Checking for EFSC (10-Q, 2024) - CIK: 0001025835
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\EFSC\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\EFSC\10-Q
No match found for EFSC in ['C:/Users/sudet/Desktop/to-be-processed\\EFSC\\10-Q']

Checking for AGM (10-Q, 2023) - CIK: 0000845877
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\AGM\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\AGM\10-Q
No match found for AGM in ['C:


Checking for AGM (10-Q, 2024) - CIK: 0000845877
📂 Looking in: C:/Users/sudet/Desktop/to-be-processed\AGM\10-Q
Path does not exist: C:/Users/sudet/Desktop/to-be-processed\AGM\10-Q
No match found for AGM in ['C:/Users/sudet/Desktop/to-be-processed\\AGM\\10-Q']


Download complete!


In [1]:
import pandas as pd 
import os
# Paths
DOWNLOAD_DIR = "C:/Users/sudet/Desktop/finai/finai/Data Collection/sec-edgar-filings"
file_path = "companies.csv"  # Update with the actual path to your CSV file

# Load CIK-to-Ticker mapping from CSV
companies_df = pd.read_csv(file_path, header=None)
cik_dict = {str(row[0]).zfill(10): row[1] for _, row in companies_df.iloc[1:].iterrows()}  # CIK -> Ticker

# Rename CIK folders to Ticker names
for cik_folder in os.listdir(DOWNLOAD_DIR):
    cik_path = os.path.join(DOWNLOAD_DIR, cik_folder)

    # Ensure it's a directory and a valid CIK
    if os.path.isdir(cik_path) and cik_folder in cik_dict:
        ticker_name = cik_dict[cik_folder]  # Get corresponding ticker
        new_path = os.path.join(DOWNLOAD_DIR, ticker_name)

        if not os.path.exists(new_path):  # Avoid overwriting existing folders
            os.rename(cik_path, new_path)
            print(f"✅ Renamed {cik_folder} ➝ {ticker_name}")
        else:
            print(f"⚠️ Skipped {cik_folder} (Ticker {ticker_name} already exists!)")
    else:
        print(f"❌ Skipped {cik_folder} (No matching ticker found)")


✅ Renamed 0000006201 ➝ AAL
✅ Renamed 0000007431 ➝ AWI
✅ Renamed 0000009092 ➝ BMI
✅ Renamed 0000012208 ➝ BIO-B
✅ Renamed 0000012659 ➝ HRB
✅ Renamed 0000016058 ➝ CACI
✅ Renamed 0000018349 ➝ SNV-PE
✅ Renamed 0000020212 ➝ CHDN
✅ Renamed 0000022356 ➝ CBSH
✅ Renamed 0000028412 ➝ CMA
✅ Renamed 0000028917 ➝ DDT
✅ Renamed 0000029644 ➝ DCI
✅ Renamed 0000030625 ➝ FLS
✅ Renamed 0000039263 ➝ CFR-PB
✅ Renamed 0000039911 ➝ GAP
✅ Renamed 0000059558 ➝ LNC-PD
✅ Renamed 0000060519 ➝ LPX
✅ Renamed 0000063276 ➝ MAT
✅ Renamed 0000066570 ➝ MNESP
✅ Renamed 0000070145 ➝ NFG
✅ Renamed 0000071691 ➝ NYT
✅ Renamed 0000082811 ➝ RRX
✅ Renamed 0000084246 ➝ RLI
✅ Renamed 0000085961 ➝ R
✅ Renamed 0000088205 ➝ SPXC
✅ Renamed 0000091388 ➝ SFD
✅ Renamed 0000094845 ➝ LEVI
✅ Renamed 0000096943 ➝ TFX
✅ Renamed 0000100826 ➝ UEPCO
✅ Renamed 0000101382 ➝ UMBFP
✅ Renamed 0000102729 ➝ VMI
✅ Renamed 0000103379 ➝ VFC
✅ Renamed 0000109380 ➝ ZIONP
✅ Renamed 0000201533 ➝ CMS-PB
✅ Renamed 0000275880 ➝ PSN
✅ Renamed 0000310522 ➝ FNMAP
✅